# DreamerV3 on Google Colab

This notebook trains [DreamerV3](https://github.com/danijar/dreamerv3) on **one** environment of your choice and then runs the full inspection suite on it: evaluation rollouts, recorded videos, world-model dreams, and a side-by-side comparison of the real environment with the model's dream.

The default task is **Atari Breakout** (`atari_breakout`). Pick a different task in section 4 if desired and run the cells top-to-bottom. To try a different task, change `TASK` and re-run from section 4 onward.

**Runtime setup:**

1. `Runtime -> Change runtime type`
2. `Hardware accelerator -> GPU` (T4 is fine for Atari Breakout)
3. Run the cells below in order

## 1. Verify the runtime

In [ ]:
!nvidia-smi || echo 'No GPU detected — switch the runtime type to GPU.'
import sys
print('Python:', sys.version)

## 2. Install DreamerV3 and shared dependencies

About 2 minutes. JAX is installed with CUDA 12 wheels so the agent trains on the Colab GPU.

In [ ]:
%pip install -q --upgrade pip
%pip install -q 'dreamerv3>=3.0.0' 'gymnasium>=0.29' tensorboard ruamel.yaml cloudpickle imageio imageio-ffmpeg matplotlib
%pip install -q --upgrade 'jax[cuda12]' 'jaxlib'

import jax
print('JAX devices:', jax.devices())

## 3. Clone the repository

The notebook imports `Trainer`, `Evaluator`, `Recorder`, and `DreamVisualizer` from this repo's `scripts/` package.

In [ ]:
import os
if not os.path.isdir('/content/rl-dreamer'):
    !git clone --depth 1 https://github.com/kuds/rl-dreamer.git /content/rl-dreamer
%cd /content/rl-dreamer

# Make `scripts.<module>` importable.
import sys
if '/content/rl-dreamer' not in sys.path:
    sys.path.insert(0, '/content/rl-dreamer')

## 4. Pick a task

Set `TASK`, `PRESET`, and `RUN_STEPS` once. Every downstream cell — training, evaluation, recording, dreams — reads these variables, so to switch envs just edit this cell and re-run from here.

The default below is **`atari_breakout`** with the `size25m` preset and 400k env steps (the standard Atari 100k benchmark with action repeat 4). On a Colab T4, that takes a few hours; for a quick smoke test, drop `RUN_STEPS` to ~20_000.

Suggested combinations (smallest → largest):

| `TASK`                                | `PRESET`   | `RUN_STEPS` | Notes                                |
|---------------------------------------|------------|------------:|--------------------------------------|
| `atari_breakout`                      | `size25m`  | 400_000     | **Default.** Atari 100k benchmark.   |
| `gym_CartPole-v1`                     | `size1m`   | 20_000      | CPU-friendly smoke test (~1 min).    |
| `crafter_reward`                      | `size25m`  | 100_000     | 2D Minecraft-like; produces dreams.  |
| `dmc_walker_walk`                     | `size12m`  | 50_000      | Continuous control from pixels.      |
| `atari_pong`                          | `size25m`  | 400_000     | Other Atari game; swap for breakout. |
| `minigrid_MiniGrid-DoorKey-6x6-v0`    | `size12m`  | 50_000      | Gridworld from pixels.               |
| `minecraft_diamond`                   | `size25m`  | 20_000      | Heavy: needs Java 8 + xvfb.          |

Set `USE_DRIVE = True` to keep checkpoints, replay, videos, and dreams across Colab sessions; otherwise everything lives under `/root/logdir` and is wiped when the runtime resets.

In [ ]:
TASK = 'atari_breakout'   # see the table above for other options
PRESET = 'size25m'
RUN_STEPS = 400_000       # Atari 100k benchmark (action repeat 4 -> 100k agent steps)
BATCH_SIZE = 16

USE_DRIVE = False  # set True to persist artifacts under Google Drive

import os
from pathlib import Path

REPO_NAME = 'rl-dreamer'
DRIVE_PROJECT_ROOT = f'/content/drive/MyDrive/Finding Theta/{REPO_NAME}/logs'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    LOGDIR_ROOT = DRIVE_PROJECT_ROOT
else:
    LOGDIR_ROOT = '/root/logdir'

os.makedirs(LOGDIR_ROOT, exist_ok=True)
os.environ['LOGDIR_ROOT'] = LOGDIR_ROOT

# Per-task logdir so multiple tasks coexist under the same root.
TASK_SLUG = TASK.replace('/', '_')
LOGDIR = Path(LOGDIR_ROOT) / TASK_SLUG
os.environ['LOGDIR'] = str(LOGDIR)

SUITE = TASK.split('_', 1)[0]
print(f'TASK   = {TASK}  (suite={SUITE})')
print(f'PRESET = {PRESET}')
print(f'STEPS  = {RUN_STEPS:,}')
print(f'LOGDIR = {LOGDIR}')

## 5. Install task-specific dependencies

Only the packages needed for `TASK` are installed. Skip this cell if you've already installed them earlier in the session.

In [ ]:
if SUITE == 'gym':
    pass  # gymnasium is already installed in section 2
elif SUITE == 'dmc':
    %pip install -q dm_control
    os.environ['MUJOCO_GL'] = 'egl'
elif SUITE == 'atari':
    %pip install -q 'gymnasium[atari,accept-rom-license]' ale-py
elif SUITE == 'crafter':
    %pip install -q crafter
elif SUITE == 'minigrid':
    %pip install -q minigrid
elif SUITE == 'minecraft':
    !apt-get -qq update
    !apt-get -qq install -y openjdk-8-jdk xvfb > /dev/null
    os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-8-openjdk-amd64'
    !java -version
    %pip install -q git+https://github.com/minerllabs/minerl@v1.0.2
    print('Minecraft note: training cells must be wrapped with xvfb-run on Colab. '
          'The Trainer class invokes the env directly; if you hit display errors, '
          'restart the runtime and run the training step from a shell with xvfb-run.')
else:
    print(f'No extra installs configured for suite {SUITE!r}.')

## 6. Launch TensorBoard (optional)

Watch live training curves while the agent learns.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "$LOGDIR_ROOT"

## 7. Train

`Trainer` builds the config (defaults + size preset + per-suite encoder/decoder keys + your `overrides`), constructs the env / agent / replay / logger, and runs the DreamerV3 training loop. It is the library form of `scripts/train.py` and the standalone `examples/0X_*.py` files.

In [ ]:
from scripts.train import Trainer

trainer = Trainer(
    task=TASK,
    preset=PRESET,
    logdir=LOGDIR,
    overrides={
        'run.steps': RUN_STEPS,
        'batch_size': BATCH_SIZE,
        'run.log_every': 60,  # seconds
    },
)
trainer.run()

## 8. Evaluate the trained checkpoint

`Evaluator` reloads the saved agent and runs deterministic rollouts. It returns a numpy array of episode returns.

In [ ]:
from scripts.evaluate import Evaluator

evaluator = Evaluator(task=TASK, preset=PRESET, logdir=LOGDIR)
returns = evaluator.run(episodes=5)

print(f'\nMean return: {returns.mean():.3f} +/- {returns.std():.3f} (n={len(returns)})')

## 9. Record rollout videos

`Recorder` writes one MP4 per episode to `<logdir>/videos/`.

In [ ]:
from scripts.record import Recorder

rec = Recorder(task=TASK, preset=PRESET, logdir=LOGDIR)
episodes = rec.record(episodes=3, fps=30)

print(f'\nSaved {len(episodes)} video(s):')
for r in episodes:
    print(f"  return={r['return']:.2f}  frames={r['frames']}  ->  {r['path']}")

In [ ]:
# Embed the first rollout video inline.
import glob
from base64 import b64encode
from IPython.display import HTML

videos = sorted(glob.glob(f'{LOGDIR}/videos/*.mp4'))
if not videos:
    print('No videos found - did the recording cell above run successfully?')
else:
    data = b64encode(open(videos[0], 'rb').read()).decode()
    HTML(f'<video width=480 controls src="data:video/mp4;base64,{data}"></video>')

## 10. World-model dreams

`DreamVisualizer.generate()` runs `agent.report()` on a batch of real observations and saves:

- **Open-loop dream / reconstruction MP4s** — the decoder's output when fed posterior latents. The first few frames are grounded by real observations; the rest are rolled out from the prior alone, so you can literally see the model predicting forward.
- **Contact-sheet PNGs** of the same frames for quick inspection.

Checkpoint selection: `DreamVisualizer` automatically loads the **best-performing** snapshot under `LOGDIR` — it cross-references `metrics.jsonl` for the step with the highest episode return and picks the matching `checkpoint_<step>.ckpt`. If no step-stamped snapshots exist it falls back to the rolling `checkpoint.ckpt`. Pass `checkpoint=` to override.

Dreams only make sense for pixel-based tasks. The default `atari_breakout` produces dreams; vector-obs tasks like `gym_CartPole-v1` will produce no output here.

In [ ]:
from scripts.visualize_dreams import DreamVisualizer

viz = DreamVisualizer(task=TASK, preset=PRESET, logdir=LOGDIR)
saved = viz.generate(fps=10)

print(f'{len(saved)} file(s) written; batch source: {viz.batch_source}')
for key, path, shape in saved:
    kind = 'video' if path.suffix == '.mp4' else 'image'
    print(f'  [{kind:5s}] {key:30s} {str(shape):20s} -> {path.name}')

In [ ]:
# Embed the first dream MP4 inline (prefers the open-loop imagination video).
import glob
from base64 import b64encode
from IPython.display import HTML

dreams = sorted(glob.glob(f'{LOGDIR}/dreams/*.mp4'))
if not dreams:
    print('No dream videos found. Pixel-based tasks only.')
else:
    preferred = [p for p in dreams if 'openl' in p.lower()] or dreams
    data = b64encode(open(preferred[0], 'rb').read()).decode()
    HTML(f'<video width=480 controls src="data:video/mp4;base64,{data}"></video>')

## 11. Dream vs. environment, side by side

`DreamVisualizer.generate_side_by_side()` pairs the **real env frames** (ground truth from the batch fed to `agent.report()`) with the **model's open-loop dream frames**, written as a single horizontally-stacked MP4. The left panel is the environment; the right panel is what DreamerV3 imagines step-by-step from the prior.

Where the two diverge is exactly where the world model is wrong. This is the most direct way to eyeball model quality. Pixel-based tasks only.

In [ ]:
sbs_path = viz.generate_side_by_side(fps=10)
if sbs_path is None:
    print('Side-by-side requires a pixel-based task and an open-loop video stream.')
else:
    print(f'Wrote {sbs_path}')
    from base64 import b64encode
    from IPython.display import HTML
    data = b64encode(open(sbs_path, 'rb').read()).decode()
    HTML(f'<video width=720 controls src="data:video/mp4;base64,{data}"></video>')

## 12. Network architecture diagrams

Three schematic matplotlib diagrams. No checkpoint required — these are static visualizations that complement [`docs/architecture.md`](../docs/architecture.md).

- `world_model` — RSSM: encoder → GRU → prior/posterior → heads.
- `imagination` — open-loop imagination rollout scored by actor/critic.
- `pipeline` — the three nested loops (env → replay → world model → actor-critic).

In [ ]:
import matplotlib.pyplot as plt
from scripts.visualize_network import DIAGRAMS, render_diagram

for name in DIAGRAMS:
    fig, _ = render_diagram(name)
    plt.show()

## 13. Next steps

- Bump `RUN_STEPS` and try a larger `PRESET` (`size25m`, `size50m`) for real results.
- Set `USE_DRIVE = True` in section 4 to persist checkpoints, videos, and dreams to Google Drive.
- Switch `TASK` to a different suite and re-run from section 4 — the rest of the notebook is task-agnostic.
- Plug in your own environment using `examples/07_custom_env.py` as a template, then add a branch to `scripts/env_builders.py` so `Trainer` / `Evaluator` / `Recorder` / `DreamVisualizer` all pick it up.
- Read [`docs/architecture.md`](../docs/architecture.md), [`docs/LEARNING.md`](../docs/LEARNING.md), and [`docs/tensorboard_guide.md`](../docs/tensorboard_guide.md) for background and metric explanations.